# 🌱 KrishiRakshak AI — EfficientNetB0 Training
**Run this notebook in Google Colab with GPU enabled**

Steps:
1. Enable GPU: Runtime → Change runtime type → T4 GPU
2. Run all cells in order
3. Download model.keras at the end
4. Put model.keras in your project's ai/ folder

In [ ]:
# Cell 1: Install dependencies
!pip install kaggle tensorflow -q
print('✅ Dependencies installed')

In [ ]:
# Cell 2: Upload Kaggle API key
# First download kaggle.json from kaggle.com → Account → Create New API Token
from google.colab import files
files.upload()  # Upload kaggle.json here
import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json
print('✅ Kaggle configured')

In [ ]:
# Cell 3: Download PlantVillage dataset
!kaggle datasets download -d emmarex/plantdisease
!unzip -q plantdisease.zip -d ./PlantVillage
import os
# Find the dataset directory
for root, dirs, files in os.walk('./PlantVillage'):
    if len(dirs) > 10:
        DATASET_PATH = root
        break
print(f'✅ Dataset found at: {DATASET_PATH}')
print(f'   Classes: {len(os.listdir(DATASET_PATH))}')

In [ ]:
# Cell 4: Train EfficientNetB0 model
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import json

IMG_SIZE = 224
BATCH_SIZE = 32

# Data generators
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    validation_split=0.2
)

train_gen = datagen.flow_from_directory(
    DATASET_PATH, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', subset='training'
)
val_gen = datagen.flow_from_directory(
    DATASET_PATH, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', subset='validation'
)

num_classes = len(train_gen.class_indices)
class_names = list(train_gen.class_indices.keys())

# Save class names
with open('class_names.json', 'w') as f:
    json.dump(class_names, f)
print(f'✅ {num_classes} classes found, saved class_names.json')

# Build model
base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base.trainable = False
inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)
model = Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3),
    tf.keras.callbacks.ModelCheckpoint('model.keras', save_best_only=True)
]

print('🚀 Phase 1: Training top layers (10 epochs)...')
model.fit(train_gen, validation_data=val_gen, epochs=10, callbacks=callbacks)

print('🚀 Phase 2: Fine-tuning entire model...')
base.trainable = True
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(train_gen, validation_data=val_gen, epochs=15, callbacks=callbacks)

print('✅ Training complete! model.keras saved.')

In [ ]:
# Cell 5: Download trained model
from google.colab import files
files.download('model.keras')
files.download('class_names.json')
print('✅ Download started! Put both files in your project ai/ folder')